# Stage 3: Nested Hyperparameter Optimization, Deep Learning & Metallurgical Residual Audit
**B.Tech Project in Metallurgical and Materials Engineering, IIT Roorkee**
**Project:** Machine-Learning Prediction of Mechanical Properties of Steels from Composition, with Inverse Alloy Design

---
### Key Highlights:
1. **Nested Cross-Validation:** Hyperparameter tuning with Optuna is performed inside outer cross-validation folds to prevent tuning leakage.
2. **Algorithm Diversity:** Comparing Tuned Random Forest, Tuned XGBoost, a custom PyTorch 2-layer MLP, and a non-parametric Gaussian Process Regressor with Matern kernel.
3. **Statistical Model Selection:** Paired comparisons across identical splits, with Nadeau-Bengio corrected 95% confidence intervals.
4. **Metallurgical Residual Audit:** Error inspection across 9 alloying elements, outlier audit of top-10 worst predicted alloys, and family breakdown (Maraging, Stainless, Tool, and Structural steels).
5. **Sanity Checks:** Permutation test (target scrambling) and single largest cluster removal test.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

# Load Stage 3 Benchmark Results
summary_df = pd.read_csv(os.path.join('..', 'results', 'stage3_model_comparison.csv'))
paired_df = pd.read_csv(os.path.join('..', 'results', 'stage3_paired_comparisons_nadeau_bengio.csv'))
worst_df = pd.read_csv(os.path.join('..', 'results', 'stage3_worst_alloys_hedged.csv'))
family_df = pd.read_csv(os.path.join('..', 'results', 'stage3_family_errors_audited.csv'))
sanity_df = pd.read_csv(os.path.join('..', 'results', 'stage3_sanity_checks.csv'))

print("Stage 3 Results Loaded Successfully.")
display(summary_df)

Stage 3 Results Loaded Successfully.


,Model,Validation_RMSE_Mean,Validation_RMSE_Std,Validation_MAE_Mean,Validation_MAE_Std,Validation_R2_Mean,Validation_R2_Std
0,Default Random Forest,158.304570,26.326628,118.579433,22.350528,0.684250,0.119099
1,Tuned Random Forest,159.569910,28.486820,119.667827,22.770746,0.685706,0.104019
2,Tuned XGBoost,160.855788,31.783373,120.848441,25.303721,0.676253,0.125939
3,Default XGBoost,162.585625,30.618096,122.906042,23.982611,0.669115,0.123892
4,PyTorch MLP,173.848822,39.762532,123.105131,27.462429,0.619100,0.168486
5,Gaussian Process Regressor,183.044256,43.767543,124.063730,24.874038,0.590255,0.148942


### 1. Paired Model Statistical Comparison
Evaluate whether differences between Tuned RF, Tuned XGB, MLP, and GP are distinguishable from zero. Resampled folds share training alloys, so the corrected interval is the one to read. No model difference survives the corrected test.

In [2]:
display(paired_df[['Comparison', 'Mean_Paired_Diff_MPa', 'Corrected_95_CI_Low', 'Corrected_95_CI_High', 'Corrected_p_value', 'Verdict']])

,Comparison,Mean_Paired_Diff_MPa,Corrected_95_CI_Low,Corrected_95_CI_High,Corrected_p_value,Verdict
0,Tuned RF vs Default RF,-1.265340,-13.856273,11.325594,0.837437,Not Distinguishable (CI Includes Zero)
1,Tuned XGB vs Default XGB,1.729837,-10.809326,14.269000,0.778298,Not Distinguishable (CI Includes Zero)
2,Tuned RF vs Tuned XGB,1.285878,-15.529539,18.101296,0.875914,Not Distinguishable (CI Includes Zero)
3,Tuned RF vs PyTorch MLP,14.278912,-22.385491,50.943315,0.429412,"Suggestive (Significant on Plain, but Fails Co..."
4,Tuned RF vs Gaussian Process,23.474346,-15.612379,62.561071,0.227138,"Suggestive (Significant on Plain, but Fails Co..."
5,Ridge with Fe vs without Fe,6.650159,-0.075896,13.376213,0.052437,"Suggestive (Significant on Plain, but Fails Co..."


### 2. Sanity Checks: Target Scrambling and Largest Cluster Removal
Verify that target scrambling destroys model performance ($R^2$ at or below zero), and that dropping the largest cluster leaves model generalization intact.

In [3]:
display(sanity_df)

,Test_Name,Description,Observed_RMSE_MPa,Observed_R2,Verdict
0,Shuffled-Target Permutation Test,Scramble target y across samples to verify mod...,320.405077,-0.173091,PASS
1,"Drop-Largest-Cluster Test (Cluster 129, size=21)",Remove the single largest alloy cluster to ver...,161.629686,0.694800,PASS


### 3. Outlier Audit: Top-10 Worst Predicted Alloys
Detailed inspection of the compositions with the largest absolute prediction errors.

In [4]:
display(worst_df[['Rank', 'Experimental_Yield_MPa', 'Predicted_Yield_MPa', 'Residual_MPa', 'Absolute_Error_MPa', 'Major_Solutes']])

,Rank,Experimental_Yield_MPa,Predicted_Yield_MPa,Residual_MPa,Absolute_Error_MPa,Major_Solutes
0,1,1164.4,1731.413221,-567.013221,567.013221,"Co: 6.20%, Cr: 5.00%, Mo: 4.99%, Ni: 11.21%, T..."
1,2,2356.4,1885.162557,471.237443,471.237443,"Co: 7.90%, Mo: 4.97%, Ni: 17.36%, Ti: 1.50%"
2,3,1870.6,1402.925192,467.674808,467.674808,"Al: 1.22%, Co: 8.61%, Cr: 11.17%, Mo: 4.99%, N..."
3,4,1233.3,1697.946291,-464.646291,464.646291,"Co: 7.00%, Mo: 4.90%, Ni: 18.40%, Ti: 1.77%"
4,5,1801.8,1347.081369,454.718631,454.718631,"Ni: 18.06%, Si: 4.27%"
5,6,1837.9,1387.608805,450.291195,450.291195,"Al: 1.19%, Co: 8.51%, Cr: 10.99%, Mo: 4.98%, N..."
6,7,1825.8,1380.205501,445.594499,445.594499,"Al: 1.21%, Co: 8.55%, Cr: 10.99%, Mo: 4.99%, N..."
7,8,1139.7,1577.162938,-437.462938,437.462938,"Co: 3.91%, Ni: 8.28%"
8,9,2157.3,1765.766307,391.533693,391.533693,"Al: 0.76%, Co: 15.35%, Mo: 1.50%, Ni: 14.67%, ..."
9,10,1779.7,1418.291196,361.408804,361.408804,"Co: 9.95%, Cr: 7.74%, Mo: 0.98%, Ni: 11.93%, T..."


### 4. Metallurgical Family Breakdown
Comparison of generalization error across Maraging, Stainless, Tool, and Structural steels. Tool steels have only 17 alloys, so that row is underpowered.

In [5]:
display(family_df)

,Alloy_Family,Sample_Count_N,Sample_Size_Flag,Mean_Exp_Yield_MPa,Yield_Strength_Std_MPa,RMSE_MPa,MAE_MPa,RMSE_over_Std_Ratio,R2
0,Stainless Steels (Cr >= 10%),147,Adequate sample size (n >= 30),1299.574830,152.307499,116.773786,83.572259,0.766698,0.412175
1,Structural & Low-Alloy Steels,107,Adequate sample size (n >= 30),1466.216822,250.994568,174.531154,132.088326,0.695358,0.516477
2,"Maraging Steels (Ni >= 12%, Co >= 5%)",41,Adequate sample size (n >= 30),1629.285366,549.632128,211.773279,186.260507,0.385300,0.851544
3,Tool & High-Carbon Steels (C >= 0.35%),17,Sample size n < 30; too small for definitive c...,1684.005882,165.747467,172.425719,145.658616,1.040292,-0.082207
